<a href="https://colab.research.google.com/github/mervefilizbaker1/NLP-Homework-03-Transformer-based-Models/blob/main/NLP_HW03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Dataset**

In [ ]:
!pip install evaluate

In [ ]:
!pip install transformers datasets peft accelerate evaluate scikit-learn -q

In [ ]:
!pip install groq -q

In [ ]:
from datasets import load_dataset
import pandas as pd
from collections import Counter
import numpy as np
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import csv
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import get_peft_model, LoraConfig, TaskType
import evaluate
import os
import time
import csv
from tqdm import tqdm
from google.colab import userdata
from groq import Groq
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
dataset = load_dataset("dair-ai/emotion")
print(dataset)

In [ ]:
print(dataset["train"][0])
print(dataset["train"][1])

In [ ]:
label_names = dataset["train"].features["label"].names
print(label_names)

In [ ]:
train_labels = dataset["train"]["label"]
for label_id, count in sorted(Counter(train_labels).items()):
    print(f"  {label_names[label_id]}: {count}")

In [ ]:
train_data = dataset["train"]
val_data   = dataset["validation"]
test_data  = dataset["test"]

print(f"Train size:      {len(train_data)}")
print(f"Validation size: {len(val_data)}")
print(f"Test size:       {len(test_data)}")

In [ ]:
label_names = dataset["train"].features["label"].names

In [ ]:
test_texts  = test_data["text"]
test_labels = test_data["label"]

### Random Baseline

In [ ]:
np.random.seed(42)

In [ ]:
num_classes = len(label_names)

In [ ]:
random_preds= np.random.randint(0, num_classes, size= len(test_labels))

In [ ]:
print("Random Baseline")
print(f"Accuracy: {accuracy_score(test_labels, random_preds)}")
print(f"F1(macro): {f1_score(test_labels, random_preds, average='macro')}")

### Majority Class Baseline

In [ ]:
majority_class = Counter(train_data["label"]).most_common(1)[0][0]
print(f"Majority class: {label_names[majority_class]}")

In [ ]:
majority_preds = [majority_class] * len(test_labels)

In [ ]:
print("Majority Class Baseline")
print(f"Accuracy: {accuracy_score(test_labels, majority_preds)}")
print(f"F1 (macro): {f1_score(test_labels, majority_preds, average='macro', zero_division=0)}")

### BOW + Logistic Regression

In [ ]:
train_texts = train_data["text"]
train_labels_list = train_data["label"]

In [ ]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_texts)
X_test  = vectorizer.transform(test_texts)

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, train_labels_list)

In [ ]:
bow_preds = lr_model.predict(X_test)

In [ ]:
print("BOW + Logistic Regression")
print(f"Accuracy:   {accuracy_score(test_labels, bow_preds)}")
print(f"F1 (macro): {f1_score(test_labels, bow_preds, average='macro')}")

In [ ]:
print(" Per-class report")
print(classification_report(test_labels, bow_preds, target_names=label_names))

In [ ]:
save_path = "/content/drive/MyDrive/NLP_HW03/predictions_baselines.csv"

with open(save_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["instance_id", "model_id", "prediction", "ground_truth"])

    for i, (rp, mp, bp, gt) in enumerate(zip(random_preds, majority_preds, bow_preds, test_labels)):
        writer.writerow([i, "random",   label_names[rp], label_names[gt]])
        writer.writerow([i, "majority", label_names[mp], label_names[gt]])
        writer.writerow([i, "bow_lr",   label_names[bp], label_names[gt]])

print("Saved!")

##**2.2 Fine-tuning Pre-trained Models**

In [ ]:
BASE_PATH = "/content/drive/MyDrive/NLP_HW03"
os.makedirs(BASE_PATH, exist_ok=True)

In [ ]:
label_names = dataset["train"].features["label"].names

In [ ]:
num_labels  = len(label_names)

In [ ]:
id2label    = {i: label_names[i] for i in range(num_labels)}
label2id    = {label_names[i]: i for i in range(num_labels)}

In [ ]:
print(f"Num labels: {num_labels}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
def get_tokenized_dataset(model_checkpoint):
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    def tokenize(batch):

        return tokenizer(batch["text"], truncation=True, max_length=128)

    tokenized = dataset.map(tokenize, batched=True)
    tokenized = tokenized.rename_column("label", "labels")
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    return tokenizer, tokenized

In [ ]:
f1_metric  = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = acc_metric.compute(predictions=predictions, references=labels)
    f1  = f1_metric.compute(predictions=predictions, references=labels, average="macro")

    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

In [ ]:
def fine_tune_with_lora(model_checkpoint, model_id):
    print(f"Fine-tuning: {model_checkpoint}")

    tokenizer, tokenized_dataset = get_tokenized_dataset(model_checkpoint)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True
    )


    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_lin", "v_lin"] if "distilbert" in model_checkpoint else ["query", "value"]
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()


    output_dir = f"{BASE_PATH}/checkpoints/{model_id}"
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )


    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics
    )

    trainer.train()

    predictions = trainer.predict(tokenized_dataset["test"])
    preds = np.argmax(predictions.predictions, axis=-1)

    print(f"\nResults({model_id}) ---")
    print(f"Accuracy:   {predictions.metrics['test_accuracy']}")
    print(f"F1 (macro): {predictions.metrics['test_f1_macro']}")

    return preds, predictions.metrics

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

In [ ]:
#DistilBert
distilbert_preds, distilbert_metrics = fine_tune_with_lora(
    "distilbert-base-uncased",
    "distilbert_lora"
)

In [ ]:
#RoBERTA
roberta_preds, roberta_metrics = fine_tune_with_lora(
    "roberta-base",
    "roberta_lora"
)

In [ ]:
save_file = f"{BASE_PATH}/predictions_transformers.csv"
test_labels = dataset["test"]["label"]

with open(save_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["instance_id", "model_id", "prediction", "ground_truth"])
    for i, (dp, rp, gt) in enumerate(zip(distilbert_preds, roberta_preds, test_labels)):
        writer.writerow([i, "distilbert_lora", label_names[dp], label_names[gt]])
        writer.writerow([i, "roberta_lora",    label_names[rp], label_names[gt]])

print(f"Saved to: {save_file}")

## **2.3 Zero-shot Classification with LLMs**

In [ ]:
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

In [ ]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say hello!"}],
    max_tokens=10
)

In [ ]:
print(response.choices[0].message.content)

In [ ]:
label_names = dataset["train"].features["label"].names

In [ ]:
def build_prompt(text):
    labels_str = ", ".join(label_names)
    return f"""You are an emotion classifier. Given a text, classify it into exactly one of these emotions: {labels_str}.

Text: "{text}"

Output ONLY the emotion label, nothing else. Do not explain, do not add punctuation."""

In [ ]:
def parse_output(raw_output):
    """
    Maps the model's raw text output to one of the valid label names.
    First tries exact match, then substring match.
    Returns "unknown" if nothing matches.
    """
    cleaned = raw_output.strip().lower()

    if cleaned in label_names:
        return cleaned

    for label in label_names:
        if label in cleaned:
            return label

    return "unknown"


In [ ]:
test_prompt = build_prompt("I am so excited about my birthday party!")
raw = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": test_prompt}],
    max_tokens=10
).choices[0].message.content

In [ ]:
print(f"Raw output : '{raw}'")
print(f"Parsed     : '{parse_output(raw)}'")

In [ ]:
test_texts  = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

In [ ]:
groq_preds    = []
unknown_count = 0

In [ ]:
for i, text in enumerate(tqdm(test_texts, desc="Groq inference")):
    try:
        raw = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": build_prompt(text)}],
            max_tokens=10
        ).choices[0].message.content

        pred = parse_output(raw)

    except Exception as e:
        print(f"Error at example {i}: {e}")
        pred = "unknown"

    if pred == "unknown":
        unknown_count += 1
        print(f"  [unknown] raw='{raw}' | text='{text[:50]}'")

    groq_preds.append(pred)
    time.sleep(0.1)

print(f"\nDone! Unknown count: {unknown_count}/{len(test_texts)}")

In [ ]:
expected_length = len(test_texts)

if len(groq_preds) > expected_length:
    print(f"Warning: groq_preds has {len(groq_preds)} elements, but expected {expected_length}. Truncating.")
    groq_preds = groq_preds[:expected_length]
elif len(groq_preds) < expected_length:
    print(f"Warning: groq_preds has {len(groq_preds)} elements, but expected {expected_length}. Padding with 'unknown'.")
    groq_preds.extend(["unknown"] * (expected_length - len(groq_preds)))

valid_indices = [i for i, p in enumerate(groq_preds) if p != "unknown"]
valid_preds   = [groq_preds[i] for i in valid_indices]
valid_labels  = [label_names[test_labels[i]] for i in valid_indices]

In [ ]:
print(f"Groq (Llama 3.1 8B) Zero-shot")
print(f"Evaluated: {len(valid_preds)}/{len(test_texts)} examples")
print(f"Accuracy:   {accuracy_score(valid_labels, valid_preds)}")
print(f"F1 (macro): {f1_score(valid_labels, valid_preds, average='macro')}")

In [ ]:
print("Per-class report")
print(classification_report(valid_labels, valid_preds, labels=label_names))

In [ ]:
save_file = f"{BASE_PATH}/predictions_groq.csv"
with open(save_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["instance_id", "model_id", "prediction", "ground_truth"])
    for i, (pred, gt) in enumerate(zip(groq_preds, test_labels)):
        writer.writerow([i, "groq_llama3", pred, label_names[gt]])

print(f"\nSaved to: {save_file}")

In [ ]:
# OLLAMA

In [ ]:
import os
print(os.path.exists(f"{BASE_PATH}/predictions_groq.csv"))

In [ ]:
OLLAMA_URL = "https://jane-unchauvinistic-harmony.ngrok-free.dev"

In [ ]:
response = requests.post(
    f"{OLLAMA_URL}/api/generate",
    json={
        "model": "gemma3:4b",
        "prompt": build_prompt("I feel so happy today!"),
        "stream": False
    },
    headers={"ngrok-skip-browser-warning": "true"},
    timeout=60
)

print(response.status_code)
print(response.json())

In [ ]:
response = requests.get(
    f"{OLLAMA_URL}/api/tags",
    headers={"ngrok-skip-browser-warning": "true"}
)
print(response.status_code)
print(response.text)

In [ ]:
test_texts  = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

In [ ]:
ollama_preds  = []
unknown_count = 0

In [ ]:
for i, text in enumerate(tqdm(test_texts, desc="Ollama inference")):
    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": "gemma3:4b",
                "prompt": build_prompt(text),
                "stream": False
            },
            headers={"ngrok-skip-browser-warning": "true"},
            timeout=60
        )
        raw  = response.json()["response"]
        pred = parse_output(raw)

    except Exception as e:
        print(f"Error at example {i}: {e}")
        pred = "unknown"

    if pred == "unknown":
        unknown_count += 1
        print(f"  [unknown] raw='{raw}' | text='{text[:50]}'")

    ollama_preds.append(pred)

print(f"\nDone! Unknown count: {unknown_count}/{len(test_texts)}")

In [ ]:
valid_indices = [i for i, p in enumerate(ollama_preds) if p != "unknown"]
valid_preds   = [ollama_preds[i] for i in valid_indices]
valid_labels  = [label_names[test_labels[i]] for i in valid_indices]

In [ ]:
print(f"Ollama (Gemma 3 4B) Zero-shot")
print(f"Evaluated: {len(valid_preds)}/{len(test_texts)} examples")
print(f"Accuracy:   {accuracy_score(valid_labels, valid_preds)}")
print(f"F1 (macro): {f1_score(valid_labels, valid_preds, average='macro')}")

In [ ]:
print("Per-class report")
print(classification_report(valid_labels, valid_preds, labels=label_names))

In [ ]:
OLLAMA_URL = "https://jane-unchauvinistic-harmony.ngrok-free.dev"

def query_ollama(text):
    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": "gemma3:4b",
            "prompt": build_prompt(text),
            "stream": False
        },
        headers={"ngrok-skip-browser-warning": "true"},
        timeout=60
    )
    return response.json()["response"]

In [ ]:
raw = query_ollama("I feel so happy today!")
print(f"Raw output: '{raw}'")

In [ ]:
print(build_prompt("test"))
print(parse_output("joy"))
print(query_ollama("I feel happy"))

In [ ]:
print(len(ollama_preds))
print(ollama_preds[:5])

In [ ]:
save_file = f"{BASE_PATH}/predictions_ollama.csv"
with open(save_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["instance_id", "model_id", "prediction", "ground_truth"])
    for i, (pred, gt) in enumerate(zip(ollama_preds, test_labels)):
        writer.writerow([i, "ollama_gemma3", pred, label_names[gt]])

print(f"Saved to: {save_file}")

In [ ]:
valid_indices = [i for i, p in enumerate(ollama_preds) if p != "unknown"]
valid_preds   = [ollama_preds[i] for i in valid_indices]
valid_labels  = [label_names[test_labels[i]] for i in valid_indices]

In [ ]:
print(f"Ollama (Gemma 3 4B) Zero-shot")
print(f"Evaluated: {len(valid_preds)}/{len(test_texts)} examples")
print(f"Accuracy:   {accuracy_score(valid_labels, valid_preds)}")
print(f"F1 (macro): {f1_score(valid_labels, valid_preds, average='macro')}")

In [ ]:
print("Per-class report")
print(classification_report(valid_labels, valid_preds, labels=label_names))

## **2.4 Baselines**

In [ ]:
train_labels = dataset["train"]["label"]
test_labels  = dataset["test"]["label"]
label_names  = dataset["train"].features["label"].names
num_classes  = len(label_names)

In [ ]:
counts = Counter(train_labels)
total  = len(train_labels)

In [ ]:
print("Class Distribution (Train)")
for label_id, count in sorted(counts.items()):
    print(f"  {label_names[label_id]}: {count} ({count/total:.2%})")

In [ ]:
probs = [counts[i] / total for i in range(num_classes)]
random_acc = sum(p**2 for p in probs)

In [ ]:
print(f" Random Baseline (theoretical)")
print(f"Accuracy: {random_acc:.4f}")
print(f"F1 (macro, uniform guess): {1/num_classes:.4f}")

In [ ]:
majority_class = counts.most_common(1)[0][0]
majority_count = counts.most_common(1)[0][1]
majority_acc   = majority_count / len(test_labels)

In [ ]:
print(f"Majority Baseline (theoretical)")
print(f"Majority class: {label_names[majority_class]}")
print(f"Accuracy: {majority_acc:.4f}")